In [ ]:
import os
import faiss
from openai import OpenAI
import numpy as np
import json
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# load json dataset
with open('recipes_dataset.json', 'r') as f:
    data = json.loads(f.read().lower())

print(len(data))
print(data[0])

In [ ]:
model = "text-embedding-3-small"
openai = OpenAI()

In [ ]:
def get_metadata(recipe):
    return {
        # "id": recipe['id'],
        "name": recipe['name'],
        "cooktime_minutes": recipe['cooktime_minutes'],
        "cuisine": recipe['cuisine'],
        "nutrition": json.dumps(recipe['nutrition']),
        "ingredients": ", ".join(recipe['ingredients']),
        # "instructions": recipe['instructions']
    }


def get_embedding(text):
    embedding = openai.embeddings.create(input=text, model=model).data[0].embedding
    return embedding


embeddings = []
ids=[]
documents = []
metadata = []

for recipe in data:
    ids.append(f"id_{recipe['id']}")
    text_to_encode = f"recipe name: {recipe['name']}; cooktime in minutes: {recipe['cooktime_minutes']}; cuisine: {recipe['cuisine']}; nutrition info: {', '.join(f'{k}- {v}' for k, v in recipe['nutrition'].items())}; ingredients: {', '.join(recipe['ingredients'])}; instructions: {recipe['instructions']}"
    documents.append(text_to_encode)
    # embedding = model.encode(text_to_encode).tolist()
    embedding = get_embedding(text_to_encode)
    embeddings.append(embedding)
    metadata.append(get_metadata(recipe))

In [ ]:
print(len(embeddings), len(embeddings[0]), len(embeddings[1]), len(embeddings[2]), len(embeddings[3]), len(embeddings[4]))

In [ ]:
# Before adding to FAISS index, we need to convert the embeddings to a numpy array of type float32
embeddings = np.array(embeddings).astype('float32')
print(embeddings.shape, embeddings.dtype, embeddings.shape[1])

index = faiss.IndexFlatIP(embeddings.shape[1])
faiss.normalize_L2(embeddings)
index.add(embeddings)


In [ ]:
# docs = [
#     "Paris is the capital and most populous city of France. The city is famed for the Eiffel Tower.",
#     "Jane Austen was an English novelist best known for 'Pride and Prejudice' and 'Sense and Sensibility'.",
#     "The Great Wall of China is a series of fortifications built to protect the ancient Chinese states.",
#     "Mount Everest, part of the Himalayas, is Earth’s highest mountain above sea level.",
#     "Mike loves the color pink more than any other color."
# ]

# client = OpenAI()

# def get_embedding(text):
#     response = client.embeddings.create(model="text-embedding-3-small", input=text)
#     return response.data[0].embedding

# embeddings = np.array([get_embedding(d) for d in docs]).astype('float32')
# print(embeddings.shape, embeddings.dtype, embeddings.shape[1])
# index = faiss.IndexFlatIP(embeddings.shape[1])
# faiss.normalize_L2(embeddings)
# index.add(embeddings)

In [ ]:
query = "i need a sweet recipe with least calories"

def retrieve(query, k=3):
    query_embedding = get_embedding(query)
    query_embedding = np.array([query_embedding]).astype('float32')
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, k)
    print(f"index shape: {indices.shape}, distances shape: {distances.shape}")
    print(indices, distances)
    print("---------------------------------------------------------------")
    results = []
    for i in range(k):
        idx = indices[0][i]
        print(f"Index: {idx}, Distance: {distances[0][i]}")
        if idx < len(documents):
            results.append({
                "document": documents[idx],
                "metadata": metadata[idx],
                "distance": distances[0][i]
            })
    print("\n------------------------------------\n")
    return results


results = retrieve(query, k=5)

for result in results:
    print(f"Document: {result['document']}")
    print(f"Metadata: {result['metadata']}")
    print(f"Distance: {result['distance']}")
    print("-----")

In [ ]:
context = "\n".join([f" - {result['document']}" for result in results])

def get_instructions(query, context):
    instructions = f"""You are a helpful assistant who answers user's queries based **only** with the facts provided in the context.

    context:
    {context}

    Question: 
    {query}

    Answer:
    """
    return instructions


print(get_instructions(query, context))

In [ ]:
chatModel = "gpt-4o-mini"

def generate_answer(instructions):
    aiResponse = openai.chat.completions.create(model=chatModel, messages=[{'role': 'system', 'content': instructions}])
    return aiResponse.choices[0].message.content.strip()


## Evaluation

In [ ]:
from datasets import Dataset

questions = [
    "I want an indian recipe made of rice and chicken",
    "Give me recipe for a dessert which has least calories",
    "What is a non-american sweet dish that I can make in a short time",
    "Provide a south american seafood dish recipe ",
    "Give me a recipe for an indian vegetarian dish"
]

ground_truths=[
    "Biryani is an indian cuisine that you can make with rice and chicken",
    "Chocolate Chip cookies is one recipe that has less calories",
    "Baklava is a middle eastern dessert that you can make in a short time",
    "Ceviche is a peruvian dish that is made with white fish",
    "Palak Paneer or Chickpea Cury are some of indian vegetarian dish"
]

rows=[]

for question, ground_truth in zip(questions, ground_truths):
    retrieval_results=retrieve(question, k=2)
    context = "\n".join([f" - {result['document']}" for result in retrieval_results])
    instruction = get_instructions(question, context)
    answer = generate_answer(instruction)
    rows.append({
        'question': question,
        'context': context,
        'answer': answer,
        'reference': ground_truth
    })

evaluation_dataset=Dataset.from_list(rows)

print(rows)

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    answer_correctness,
    answer_relevancy,
    faithfulness,
    context_precision,
    context_recall,
)

scores = evaluate(
    evaluation_dataset,
    metrics=[
        answer_correctness,
        answer_relevancy,
        faithfulness,
        context_precision,
        context_recall,
    ],
)

print(scores)
